In [22]:
import numpy as np
import pandas as pd
import gymnasium as gym
# import pandas_ta as ta
import gym_trading_env
from sb3_contrib import RecurrentPPO
import wandb
from gym_trading_env.wrapper import DiscreteActionsWrapper
from wandb.integration.sb3 import WandbCallback
from stable_baselines3 import DQN
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from stable_baselines3.common.monitor import Monitor 

# Import de nos propres fichiers
import reward as reward_functions
import features as features


In [23]:
from typing import Callable

def linear_schedule(initial_value: float) -> Callable[[float], float]:
    def func(progress_remaining: float) -> float:
        return progress_remaining * initial_value
    return func

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

In [24]:
os.environ["WANDB_ERROR_REPORTING"] = "False" 
os.environ["WANDB_CONSOLE"] = "off"

# Model 1

On essaye un model basique, avec preprocess basic

In [25]:
base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_basic,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model1 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model1.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model1.save("DQN_1")


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


global_step,▁▁▁▁▁▁██████
rollout/ep_len_mean,▁█
rollout/ep_rew_mean,█▁
rollout/exploration_rate,█▁
time/fps,█▁
train/learning_rate,▁▁
train/loss,▁█
global_step,133814
rollout/ep_len_mean,16726.75
rollout/ep_rew_mean,-20.14922
rollout/exploration_rate,0.57626


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/amahshfq\DQN_1
Market Return : 39.94%   |   Portfolio Return : -98.48%   |   
Market Return : 904.05%   |   Portfolio Return : -100.00%   |   
Market Return : 103.26%   |   Portfolio Return : -100.00%   |   
Market Return : 44.45%   |   Portfolio Return : -98.19%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.57e+04 |
|    ep_rew_mean      | -19.6    |
|    exploration_rate | 0.802    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1035     |
|    time_elapsed     | 60       |
|    total_timesteps  | 62611    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 7.19e-05 |
|    n_updates        | 15627    |
----------------------------------
Market Return : 163.91%   |   Portfolio Return : -100.00%   |   
Market Return : 677.81%   |  

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [26]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 
    
    while not done:
        action, _states = model1.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 27.70%   |   Portfolio Return : -27.34%   |   
Episode 1 loggé : yfinance-STOXX50-1h.pkl -> PnL: -27.34%
Market Return : 163.91%   |   Portfolio Return : -99.93%   |   
Episode 2 loggé : binance-DOGEEUR-1h.pkl -> PnL: -99.93%
Market Return : 677.81%   |   Portfolio Return : -99.88%   |   
Episode 3 loggé : binance-BTCUSD-1h.pkl -> PnL: -99.88%
Market Return : 39.94%   |   Portfolio Return : -52.57%   |   
Episode 4 loggé : yfinance-AAPL-1h.pkl -> PnL: -52.57%
Market Return :  5.27%   |   Portfolio Return : -83.69%   |   
Episode 5 loggé : yfinance-EURUSD-1h.pkl -> PnL: -83.69%
Market Return : 103.26%   |   Portfolio Return : -89.04%   |   
Episode 6 loggé : yfinance-GOLDUSD-1h.pkl -> PnL: -89.04%
Market Return : 44.45%   |   Portfolio Return :  1.47%   |   
Episode 7 loggé : yfinance-S&P500-1h.pkl -> PnL: 1.47%
Market Return : 677.81%   |   Portfolio Return : -99.87%   |   
Episode 8 loggé : binance-BTCUSD-1h.pkl -> PnL: -99.87%
Market Return : 163.91%   |   Portfolio R

eval/cr_pourcent,▆▁▁▄▂▂█▁▁▆▇▁▆▂▂▇▂▁▇▂▁▁▅▇▁▇▁▅▁▄
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,▆▁▁▄▂▂█▁▁▆▇▁▆▂▂▇▂▁▇▂▁▁▅▇▁▇▁▅▁▄
global_step,▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇██████
rollout/ep_len_mean,▁▆█▆▅▄▄▃▄▃▄▄▄▄▄▄▅▅▄▄
rollout/ep_rew_mean,▃▁▂▃▄▅▆▆▇▇▇▇▇███████
rollout/exploration_rate,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,▅████▆▅▆▆▆▅▄▃▂▂▁▁▁▁▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▄▂█▂▁▅▄▂▂▁▅▁▄▁▃▅▂▂▃▁
eval/cr_pourcent,-58.47534


# Model 2

On essaye avec preprocess dqn

In [30]:
base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_dqn,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])

obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model2 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model2.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model2.save("DQN_2")


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/d4j3rjlg\DQN_1
Market Return : 38.69%   |   Portfolio Return : -97.40%   |   
Market Return :  5.60%   |   Portfolio Return : -99.49%   |   
Market Return : 22.46%   |   Portfolio Return : -99.56%   |   
Market Return : 98.71%   |   Portfolio Return : -100.00%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 5.83e+03 |
|    ep_rew_mean      | -7.04    |
|    exploration_rate | 0.926    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1561     |
|    time_elapsed     | 14       |
|    total_timesteps  | 23328    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 0.000269 |
|    n_updates        | 5806     |
----------------------------------
Market Return : 208.65%   |   Portfolio Return : -100.00%   |   
Market Return : 46.62%   |   Por

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [31]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 
    
    while not done:
        action, _states = model2.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 22.46%   |   Portfolio Return : -51.18%   |   
Episode 1 loggé : yfinance-STOXX50-1h.pkl -> PnL: -51.18%
Market Return : 46.62%   |   Portfolio Return : -58.39%   |   
Episode 2 loggé : yfinance-AAPL-1h.pkl -> PnL: -58.39%
Market Return :  5.60%   |   Portfolio Return : -50.22%   |   
Episode 3 loggé : yfinance-CAC40-1h.pkl -> PnL: -50.22%
Market Return : 677.17%   |   Portfolio Return : -100.00%   |   
Episode 4 loggé : binance-BTCUSD-1h.pkl -> PnL: -100.00%
Market Return : 1018.04%   |   Portfolio Return : -99.99%   |   
Episode 5 loggé : binance-ETHUSD-1h.pkl -> PnL: -99.99%
Market Return : 208.65%   |   Portfolio Return : -99.86%   |   
Episode 6 loggé : binance-DOGEEUR-1h.pkl -> PnL: -99.86%
Market Return : 38.69%   |   Portfolio Return : -60.83%   |   
Episode 7 loggé : yfinance-S&P500-1h.pkl -> PnL: -60.83%
Market Return : 22.46%   |   Portfolio Return : -48.37%   |   
Episode 8 loggé : yfinance-STOXX50-1h.pkl -> PnL: -48.37%
Market Return : 208.65%   |   Portfol

eval/cr_pourcent,█▇█▁▁▁▆█▁▁▆▆▁▁▃▄▅▁▆▃▁▁▁▆▅▁▆▄▆▁
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,█▇█▁▁▁▆█▁▁▆▆▁▁▃▄▅▁▆▃▁▁▁▆▅▁▆▄▆▁
global_step,▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
rollout/ep_len_mean,▁█▇▇▆▇▇▇▇▇▇█▇█▇█▇█▇▇
rollout/ep_rew_mean,█▁▂▃▄▄▅▆▆▆▇▇▇▇▇▇▇▇██
rollout/exploration_rate,█▅▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▃▁▃▁▂▁▁▁▂▁▄▂▂▂█▂▂▄▂
eval/cr_pourcent,-99.98073


# Model 3

On essaye avec preprocess finance

In [32]:
base_env = gym.make(

    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])

obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model3 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model3.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model3.save("DQN_3")


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/x3wh0ubk\DQN_1
Market Return : 27.57%   |   Portfolio Return : -99.57%   |   
Market Return :  5.19%   |   Portfolio Return : -100.00%   |   
Market Return : 10.31%   |   Portfolio Return : -99.67%   |   
Market Return : 209.83%   |   Portfolio Return : -100.00%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.59e+04 |
|    ep_rew_mean      | -20.7    |
|    exploration_rate | 0.799    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1111     |
|    time_elapsed     | 57       |
|    total_timesteps  | 63609    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 1.43e-05 |
|    n_updates        | 15877    |
----------------------------------
Market Return : 40.24%   |   Portfolio Return : -98.83%   |   
Market Return : 680.05%   |   Po

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [33]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 

    
    while not done:
        action, _states = model3.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 680.05%   |   Portfolio Return : -100.00%   |   
Episode 1 loggé : binance-BTCUSD-1h.pkl -> PnL: -100.00%
Market Return : 102.45%   |   Portfolio Return : -92.23%   |   
Episode 2 loggé : yfinance-GOLDUSD-1h.pkl -> PnL: -92.23%
Market Return : 27.57%   |   Portfolio Return : -70.53%   |   
Episode 3 loggé : yfinance-STOXX50-1h.pkl -> PnL: -70.53%
Market Return : 209.83%   |   Portfolio Return : -100.00%   |   
Episode 4 loggé : binance-DOGEEUR-1h.pkl -> PnL: -100.00%
Market Return : 902.89%   |   Portfolio Return : -100.00%   |   
Episode 5 loggé : binance-ETHUSD-1h.pkl -> PnL: -100.00%
Market Return : 44.32%   |   Portfolio Return : -65.50%   |   
Episode 6 loggé : yfinance-S&P500-1h.pkl -> PnL: -65.50%
Market Return : 10.31%   |   Portfolio Return : -70.81%   |   
Episode 7 loggé : yfinance-CAC40-1h.pkl -> PnL: -70.81%
Market Return : 40.24%   |   Portfolio Return : -13.85%   |   
Episode 8 loggé : yfinance-AAPL-1h.pkl -> PnL: -13.85%
Market Return : 40.24%   |   Port

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Market Return : 40.24%   |   Portfolio Return : -30.51%   |   
Episode 30 loggé : yfinance-AAPL-1h.pkl -> PnL: -30.51%


eval/cr_pourcent,▁▂▃▁▁▄▃█▅▃▁▆▄▁▁▁▂▁▆▅▃▃▁▁▂▁▅▂▃▇
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,▁▂▃▁▁▄▃█▅▃▁▆▄▁▁▁▂▁▆▅▃▃▁▁▂▁▅▂▃▇
global_step,▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇██
rollout/ep_len_mean,▃▁▂▆▆█▇▇▆█▇▆█▆▇▇▇▇▇
rollout/ep_rew_mean,▁▂▃▂▄▄▅▆▆▆▇▇▇▇█████
rollout/exploration_rate,█▆▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,█▆▅▃▂▂▁▁▂▄▄▅▅▅▆▆▇▇▇
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▁▃▂▁▃▂▆▂▆▂▃▅▂▂▅█▆▄█
eval/cr_pourcent,-30.50725


# Model 4

On essaye avec feature preprocess finance finale

In [34]:
base_env = gym.make(

    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])

obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model4 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model4.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model4.save("DQN_4")


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/42i2avun\DQN_1
Market Return : 26.64%   |   Portfolio Return : -99.42%   |   
Market Return : 39.26%   |   Portfolio Return : -98.29%   |   
Market Return : 699.56%   |   Portfolio Return : -100.00%   |   
Market Return :  5.73%   |   Portfolio Return : -100.00%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.66e+04 |
|    ep_rew_mean      | -19      |
|    exploration_rate | 0.79     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1099     |
|    time_elapsed     | 60       |
|    total_timesteps  | 66442    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 2.03e-05 |
|    n_updates        | 16585    |
----------------------------------
Market Return : 469.95%   |   Portfolio Return : -100.00%   |   
Market Return : 42.46%   |   P

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [35]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 

    
    while not done:
        action, _states = model4.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 104.07%   |   Portfolio Return : -32.92%   |   
Episode 1 loggé : yfinance-GOLDUSD-1h.pkl -> PnL: -32.92%
Market Return : 39.26%   |   Portfolio Return : 40.90%   |   
Episode 2 loggé : yfinance-AAPL-1h.pkl -> PnL: 40.90%
Market Return :  5.73%   |   Portfolio Return : -70.40%   |   
Episode 3 loggé : yfinance-EURUSD-1h.pkl -> PnL: -70.40%
Market Return : 959.21%   |   Portfolio Return : -98.99%   |   
Episode 4 loggé : binance-ETHUSD-1h.pkl -> PnL: -98.99%
Market Return : 469.95%   |   Portfolio Return : -100.00%   |   
Episode 5 loggé : binance-DOGEEUR-1h.pkl -> PnL: -100.00%
Market Return : 42.46%   |   Portfolio Return :  0.01%   |   
Episode 6 loggé : yfinance-S&P500-1h.pkl -> PnL: 0.01%
Market Return : 26.64%   |   Portfolio Return : -40.84%   |   
Episode 7 loggé : yfinance-STOXX50-1h.pkl -> PnL: -40.84%
Market Return :  9.68%   |   Portfolio Return : -60.37%   |   
Episode 8 loggé : yfinance-CAC40-1h.pkl -> PnL: -60.37%
Market Return :  9.68%   |   Portfolio Ret

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode 30 loggé : binance-BTCUSD-1h.pkl -> PnL: -97.92%


eval/cr_pourcent,▄▇▂▁▁▅▃▃▄▂▄▁▁▁▅▄▃▆▁▃▂▁█▅▂▁▂▅▁▁
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,▄▇▂▁▁▅▃▃▄▂▄▁▁▁▅▄▃▆▁▃▂▁█▅▂▁▂▅▁▁
global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇██████
rollout/ep_len_mean,▂▁▁▅▇▆▄▇█▇▆█▆▆▅▆▇▆▆
rollout/ep_rew_mean,▁▁▂▃▄▅▆▆▆▇▇▇▇██████
rollout/exploration_rate,█▆▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,█▇▆▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▁▁▁▁▂▁▁▁▂▂▁▂▁▁▂▁█▂▁
eval/cr_pourcent,-97.91786


# Model 5

On essaye avec une fonction de récompense en log

In [36]:
base_env = gym.make(

    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_functions.reward_log_returns
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])

obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model5 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model5.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model5.save("DQN_5")


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/qc08bybm\DQN_1
Market Return : 39.26%   |   Portfolio Return : -99.10%   |   
Market Return : 26.64%   |   Portfolio Return : -99.60%   |   
Market Return : 469.95%   |   Portfolio Return : -100.00%   |   
Market Return : 42.46%   |   Portfolio Return : -97.90%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.34e+04 |
|    ep_rew_mean      | -18      |
|    exploration_rate | 0.831    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1095     |
|    time_elapsed     | 48       |
|    total_timesteps  | 53430    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 0.000107 |
|    n_updates        | 13332    |
----------------------------------
Market Return : 104.07%   |   Portfolio Return : -100.00%   |   
Market Return :  9.68%   |   Po

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [37]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 

    
    while not done:
        action, _states = model5.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 959.21%   |   Portfolio Return : -98.51%   |   
Episode 1 loggé : binance-ETHUSD-1h.pkl -> PnL: -98.51%
Market Return : 39.26%   |   Portfolio Return : -42.70%   |   
Episode 2 loggé : yfinance-AAPL-1h.pkl -> PnL: -42.70%
Market Return : 42.46%   |   Portfolio Return : 28.37%   |   
Episode 3 loggé : yfinance-S&P500-1h.pkl -> PnL: 28.37%
Market Return : 104.07%   |   Portfolio Return : -40.29%   |   
Episode 4 loggé : yfinance-GOLDUSD-1h.pkl -> PnL: -40.29%
Market Return : 699.56%   |   Portfolio Return : -88.04%   |   
Episode 5 loggé : binance-BTCUSD-1h.pkl -> PnL: -88.04%
Market Return :  9.68%   |   Portfolio Return : -17.30%   |   
Episode 6 loggé : yfinance-CAC40-1h.pkl -> PnL: -17.30%
Market Return : 26.64%   |   Portfolio Return :  4.66%   |   
Episode 7 loggé : yfinance-STOXX50-1h.pkl -> PnL: 4.66%
Market Return :  9.68%   |   Portfolio Return : -29.23%   |   
Episode 8 loggé : yfinance-CAC40-1h.pkl -> PnL: -29.23%
Market Return : 26.64%   |   Portfolio Return 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Market Return : 104.07%   |   Portfolio Return : -10.02%   |   
Episode 30 loggé : yfinance-GOLDUSD-1h.pkl -> PnL: -10.02%


eval/cr_pourcent,▁▂▄▃▁▃▄▃▃▁▂▃▄▁█▃▁▃▃▃▂▂▂▂▃▃▄▁▄▃
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,▁▂▄▃▁▃▄▃▃▁▂▃▄▁█▃▁▃▃▃▂▂▂▂▃▃▄▁▄▃
global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
rollout/ep_len_mean,▃▁▆▅▃▅▇▇▆▇█▇▇█████▇█
rollout/ep_rew_mean,▁▃▂▃▄▅▅▆▆▇▇▇▇▇██████
rollout/exploration_rate,█▇▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,██▆▅▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▃▂▃▁▄▃▁▂▃▂▁█▄▁▄▅▁▁▃
eval/cr_pourcent,-10.01597


# Model 6

On essaye avec une fonction de récompense rapport market

In [38]:
base_env = gym.make(

    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_functions.reward_log_returns
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])

obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model6 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model6.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model6.save("DQN_6")


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/sqay27l7\DQN_1
Market Return : 39.26%   |   Portfolio Return : -96.43%   |   
Market Return : 26.64%   |   Portfolio Return : -99.53%   |   
Market Return : 959.21%   |   Portfolio Return : -100.00%   |   
Market Return : 699.56%   |   Portfolio Return : -100.00%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.45e+04 |
|    ep_rew_mean      | -28.3    |
|    exploration_rate | 0.689    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1081     |
|    time_elapsed     | 90       |
|    total_timesteps  | 98154    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 2.09e-05 |
|    n_updates        | 24513    |
----------------------------------
Market Return : 469.95%   |   Portfolio Return : -100.00%   |   
Market Return :  9.68%   |   

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


In [39]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = base_env.name 

    
    while not done:
        action, _states = model6.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if done:
        history = base_env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Market Return : 42.46%   |   Portfolio Return : 30.11%   |   
Episode 1 loggé : yfinance-S&P500-1h.pkl -> PnL: 30.11%
Market Return : 26.64%   |   Portfolio Return : -23.74%   |   
Episode 2 loggé : yfinance-STOXX50-1h.pkl -> PnL: -23.74%
Market Return : 42.46%   |   Portfolio Return : 19.44%   |   
Episode 3 loggé : yfinance-S&P500-1h.pkl -> PnL: 19.44%
Market Return :  5.73%   |   Portfolio Return : -65.98%   |   
Episode 4 loggé : yfinance-EURUSD-1h.pkl -> PnL: -65.98%
Market Return :  9.68%   |   Portfolio Return : -33.54%   |   
Episode 5 loggé : yfinance-CAC40-1h.pkl -> PnL: -33.54%
Market Return : 469.95%   |   Portfolio Return : -99.84%   |   
Episode 6 loggé : binance-DOGEEUR-1h.pkl -> PnL: -99.84%
Market Return : 959.21%   |   Portfolio Return : -97.60%   |   
Episode 7 loggé : binance-ETHUSD-1h.pkl -> PnL: -97.60%
Market Return : 39.26%   |   Portfolio Return : -5.28%   |   
Episode 8 loggé : yfinance-AAPL-1h.pkl -> PnL: -5.28%
Market Return : 104.07%   |   Portfolio Return 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode 30 loggé : yfinance-S&P500-1h.pkl -> PnL: 23.03%


eval/cr_pourcent,█▅▇▃▅▁▁▆█▁▃▁▆▅▅▁▁▅▃▁▁▄▅▇▅▃▇▁▅█
eval/episode,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/final_value,█▅▇▃▅▁▁▆█▁▃▁▆▅▅▁▁▅▃▁▁▄▅▇▅▃▇▁▅█
global_step,▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
rollout/ep_len_mean,██▅▃▂▂▁▂▃▃▂▂▂▂▂▂▂▂▂
rollout/ep_rew_mean,▁▂▄▅▆▆▇▇▇▇▇████████
rollout/exploration_rate,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/fps,▃▆▇████▅▄▃▃▂▂▂▂▁▁▁▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▂▃█▁▁▂▂▃▁▁▁▁▁▃▃▃▂▁▃
eval/cr_pourcent,23.02535
